## Figure 3

### cleaned_BioBERT_data.csv

In [1]:
import pandas as pd

new_NER_gene_list = pd.read_csv("../../results_openalex/03_gene_extraction/cleaned_BioBERT_data.csv")
old_NER_gene_list = pd.read_csv("../../file_merging/Previous/cleaned_BioBERT_data.csv")

In [2]:
genes = pd.read_csv('../../notebooks/03_gene_extraction/genes_CIVIC.csv')
mapping = genes.assign(**{'Gene ID': genes['Gene ID'].astype(str)}) \
               .set_index('Gene ID')['Gene Name'] \
               .to_dict()
new_NER_gene_list = new_NER_gene_list.rename(columns=mapping)
new_NER_gene_list

,PaperId,PaperTitle,Citations,CoFoS,Authors,Abstract,Language,PubYear,PubDate,BioBERT,...,YES1,YTHDF2,ZEB1,ZFP36L1,ZHX2,ZNF217,ZNF292,ZNF554,ZRSR2,Sum_Gene_Mentions
0,7106250716,PVNDMVLiver Circuit Drives Hepatocellular Carc...,NaN,50738837|2778019345|71924100|96525457|27793434...,"Chi, Dongmei",Hepatocellular carcinoma (HCC) and depression ...,en,2026,2026-11-20,9,...,0,0,0,0,0,0,0,0,0,1
1,7125254527,Force coordination distinguishes epithelial an...,NaN,198826908|54166955|95444343|126749454|13773824...,"J. Illescas Diaz, Roberto Mayor",Collective cell migration is essential for dev...,en,2026,2026-03-02,1,...,0,0,0,0,0,0,0,0,0,1
2,7118668739,Influence of the interaction between the Notch...,NaN,71924100|137061746|500558357|170493617|1263220...,"QI Guibin, GAO Jianbu, ZHANG Minglei, ZHANG Yo...",Objective To investigate the influence of the ...,en,2026,2026-02-01,"5, 6, 10",...,0,0,0,0,0,0,0,0,0,3
3,7119160011,Screening and biological function analysis of ...,NaN,152724338|86803240|104317684|2778019345|140364...,"SUN Chenggong, TIAN Ying, JIANG Feng",Objective To investigate the differentially ex...,en,2026,2026-02-01,21,...,0,0,0,0,0,0,0,0,0,1
4,7125353320,IP6K3 laat metabole aanpassing van nierkankerc...,NaN,2777701055|2780674031|162317418|8891405|121608...,Silvia Rivis,"In the last years, immunotherapy has emerged a...",en,2026,2026-01-30,"4, 1",...,0,0,0,0,0,0,0,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27064,7124461420,Can idebenone protect gastric tissue against o...,4281250025|2761843869|3137378678|2073003655|30...,2776151105|71924100|2781263782|2777403772|2778...,"C. Kaya, Tugba Nurcan Yuksel, ERDEM TOKTAY, El...",This study aims to investigate the antiulcer e...,en,2025,2025-01-01,1,...,0,0,0,0,0,0,0,0,0,1
27065,7124515032,Co-targeting of cancer glycolytic metabolism a...,NaN,2777802072|2776194525|502942594|2780394083|185...,Ana Catarina da Silva Pacheco,O glioblastoma (GBM) um tumor cerebral primrio...,en,2025,2025-01-01,9,...,0,0,0,0,0,0,0,0,0,1
27066,7124877087,Natural Products Targeting Oncogenes and Tumor...,NaN,179185449|2778264664|41091548|121608353|502942...,"Md Faiyaz, Chand Yadav, Dr.Pushpendra Kumar, K...",Natural products are a vital source of antican...,en,2025,2025-01-01,2,...,0,0,0,0,0,0,0,0,0,1
27067,7125091241,18F-PSMA PET/CT vizsglat prosztatark biokmiai ...,NaN,71924100|2780192828|2779466945|126322002|29456...,"Domonkos Ndasdy-Horvth, Mrton Piroska, Sndor C...",Bevezets: A radiklis prostatectomia utni biokm...,en,2025,2025-01-01,7,...,0,0,0,0,0,0,0,0,0,1


In [3]:
import warnings
warnings.filterwarnings("ignore")

keys = ['PaperId', 'PaperTitle', 'Citations', 'CoFoS', 'Authors',
        'Abstract', 'Language', 'PubYear', 'PubDate']

merged = old_NER_gene_list.merge(
    new_NER_gene_list,
    on=keys,
    how='outer',
    suffixes=('_old', '_new')
)

NER_gene_list = merged[keys].copy()

# 👉 colonne NON chiave nei dataframe originali
old_cols = set(old_NER_gene_list.columns) - set(keys)
new_cols = set(new_NER_gene_list.columns) - set(keys)

all_cols = old_cols.union(new_cols)

for col in all_cols:
    old_col = f'{col}_old'
    new_col = f'{col}_new'

    if old_col in merged.columns and new_col in merged.columns:
        if col == 'Sum_Gene_Mentions':
            # fai la somma invece del OR logico
            NER_gene_list[col] = (
                merged[old_col].fillna(0) + merged[new_col].fillna(0)
            ).astype(int)
        else:
            # presenti in entrambi → OR logico
            NER_gene_list[col] = (
                merged[old_col].fillna(0).astype(bool) |
                merged[new_col].fillna(0).astype(bool)
            ).astype(int)

    elif old_col in merged.columns:
        NER_gene_list[col] = merged[old_col].fillna(0).astype(int)

    elif new_col in merged.columns:
        NER_gene_list[col] = merged[new_col].fillna(0).astype(int)

    else:
        NER_gene_list[col] = merged[col].fillna(0).astype(int)

NER_gene_list.to_csv("../../file_merging/Updated/cleaned_BioBERT_data.csv", index=False)

print(f'Shape after merge and process: {NER_gene_list.shape}')

Shape after merge and process: (335817, 744)


## Figure 4

In [21]:
old_cancer_counts_df = pd.read_csv('../../file_merging/Previous/cancer_counts_of_all_cancer_types.csv')
new_cancer_counts_df = pd.read_csv('../../results_openalex/04_categorization/cancer_counts_of_all_cancer_types.csv')

cancer_counts_df = pd.merge(old_cancer_counts_df, new_cancer_counts_df, how='outer', on=['cancer_type', 'final_parent']).fillna(0.0)

cancer_counts_df['count'] = cancer_counts_df['count_x'] + cancer_counts_df['count_y']
cancer_counts_df = cancer_counts_df.drop(columns=['count_x', 'count_y'])

cancer_df_length = len(cancer_counts_df) #len(cancer_df)
cancer_category_occurrences = cancer_counts_df.groupby("final_parent", as_index=False)["count"].sum()
cancer_category_occurrences = cancer_category_occurrences.sort_values(by="count", ascending=False)
other_cancers = cancer_category_occurrences[cancer_category_occurrences["count"] < 300]
other_cancers_sum = other_cancers["count"].sum()
cancer_category_occurrences = cancer_category_occurrences[cancer_category_occurrences["count"] >= 300]
other_cancers_row = pd.DataFrame({"final_parent": ["other cancers"], "count": [other_cancers_sum]})
cancer_category_occurrences = pd.concat([cancer_category_occurrences, other_cancers_row], ignore_index=True)
total_mentions = cancer_df_length
cancer_category_occurrences["percentage"] = ((cancer_category_occurrences["count"] / total_mentions) * 100).round(2)

print("\nSummed Cancer Category Occurrences:")
print("Length of category dataset:", len(cancer_category_occurrences))
print(cancer_category_occurrences)

cancer_category_occurrences.to_csv("../../file_merging/Updated/cancer_category_occurrences_with_percentages.csv", index=False)
print("\nCSV file saved successfully as 'cancer_category_occurrences_with_percentages.csv'")


Summed Cancer Category Occurrences:
Length of category dataset: 50
                     final_parent    count  percentage
0                   breast cancer  82869.0    27170.16
1                    colon cancer  37142.0    12177.70
2                 prostate cancer  33888.0    11110.82
3                     lung cancer  27590.0     9045.90
4               pancreatic cancer  18361.0     6020.00
5                          glioma  17934.0     5880.00
6                    liver cancer  16234.0     5322.62
7                  ovarian cancer  13794.0     4522.62
8                        melanoma  10852.0     3558.03
9              endometrial cancer   8354.0     2739.02
10                  cervix cancer   7433.0     2437.05
11                       leukemia   7358.0     2412.46
12                 thyroid cancer   7279.0     2386.56
13           head and neck cancer   5012.0     1643.28
14                   renal cancer   4889.0     1602.95
15                 bladder cancer   4883.0     1600.

## Figure 5

In [1]:
import pandas as pd

old_figure_df = pd.read_csv('../../file_merging/Previous/full_df_with_umap_for_scatter_plot.csv')
new_figure_df = pd.read_csv('../../results_openalex/04_04_landscape/full_df_with_umap_for_scatter_plot.csv')

figure_df = pd.concat([old_figure_df, new_figure_df], axis=0, ignore_index=True)
figure_df.to_csv('../../file_merging/Updated/full_df_with_umap_for_scatter_plot.csv', index=False)
figure_df

,PaperId,PaperTitle,Abstract,PubYear,Study_design,Study_design_clean,UMAP_1,UMAP_2,Study Type,Probability
0,4405900528,Tissue Prior to the Initial Hematoxylin-Eosin ...,"Small biopsies are used for histologic, immuno...",2024.0,In vitro study,In vitro study,7.001095,6.870371,NaN,NaN
1,4405900803,Detection of urine circulating tumor DNA using...,Aim: This study aims to evaluate the feasibili...,2024.0,Clinical study,Clinical study,5.984958,7.226529,NaN,NaN
2,4405901040,Evaluation of a Novel PLGA-HA-Based Drug Deliv...,Colorectal carcinoma (CRC) is a very important...,2024.0,In vitro study,In vitro study,8.176593,-0.101430,NaN,NaN
3,4405919948,Gasdermin D regulates the activation of EGFR i...,Gasdermin D (GSDMD) is a key effector molecule...,2024.0,In vitro study,In vitro study,5.729273,0.228602,NaN,NaN
4,4405924568,Silencing of Epidermal Growth Factor-like Doma...,Ovarian cancer (OC) is the second most common ...,2024.0,In vitro study,In vitro study,6.296736,1.020107,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
214219,7116128796,Expression of endothelin-1 in unaffected colon...,Introduction: Endothelin-1 is a vasoconstricto...,NaN,Clinical study,Clinical study,6.581621,7.491122,-,NaN
214220,7117235300,Time-dependent exposure to venetoclax induces ...,Aim: Neuroblastoma is among the most widely di...,NaN,In vitro study,In vitro study,6.145976,0.541755,-,NaN
214221,7118161155,USP3 Promotes Glioma Progression by Stabilizin...,Glioma is an aggressive primary brain tumor wi...,NaN,In vitro study,In vitro study,5.544490,2.513251,-,NaN
214222,7120596868,Translational study of IL-33 expression and it...,Triple-negative breast cancer (TNBC) is an agg...,NaN,Clinical study,Clinical study,7.358807,7.179100,-,NaN


## Figure 7

In [15]:
import pandas as pd

old_CIVIC_ncit_df_finalparent = pd.read_csv("../../file_merging/Previous/CIVIC_ncit_df_finalparent_treatmentcategory.csv")
new_CIVIC_ncit_df_finalparent = pd.read_csv("../../results_openalex/04_categorization/CIVIC_ncit_df_finalparent_treatmentcategory.csv")
print(f'Previous shape: {old_CIVIC_ncit_df_finalparent.shape}')
print(f'New shape: {new_CIVIC_ncit_df_finalparent.shape}')

CIVIC_ncit_df_finalparent = pd.concat([old_CIVIC_ncit_df_finalparent, new_CIVIC_ncit_df_finalparent])
l = len(CIVIC_ncit_df_finalparent)
CIVIC_ncit_df_finalparent = CIVIC_ncit_df_finalparent.drop_duplicates()
print("Duplicates: ", l - len(CIVIC_ncit_df_finalparent))
print("Updated shape: ", CIVIC_ncit_df_finalparent.shape)
CIVIC_ncit_df_finalparent

Previous shape: (565, 24)
New shape: (622, 24)
Duplicates:  184
Updated shape:  (1003, 24)


,id,name,therapyUrl,ncitId,therapyAliases_old,filteredAliases,therapyAliases,original_alias_count,new_alias_count,definitions,...,parent_3,parent_4,parent_5,parent_6,parent_7,parent_8,parent_9,final_parent,final_parent_count,parent_treatment_category
0,302,"2,4-pyrimidinediamine",NaN,NaN,[],[],[],0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,4-pyrimidinediamine",1,Other therapy
1,5876,3-Dimensional Conformal Radiation Therapy,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...,C16035,"['3D CRT', '3D-CRT', '3D Conformal', '3D Radio...",[],"['3D CRT', '3D-CRT', '3D Conformal', '3D Radio...",10,10,A procedure that uses a computer to create a 3...,...,Clinical Intervention or Procedure,Clinical or Research Activity,Activity,NaN,NaN,NaN,NaN,Radiation Therapy,2,Radiation therapy
2,23383,4'-(9-acridinylamino)methanesulfon-m-anisidide,NaN,NaN,[],[],[],0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4'-(9-acridinylamino)methanesulfon-m-anisidide,1,Other therapy
3,18995,5-Fluorouracil/Salicylic Acid Topical Solution,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...,C97131,"['LAS 41005', 'Actikerall', 'Low-dose 5-Fluoro...",[],"['LAS 41005', 'Actikerall', 'Low-dose 5-Fluoro...",3,3,A topical formulation containing 0.5 % of anti...,...,"Drug, Food, Chemical or Biomedical Material",NaN,NaN,NaN,NaN,NaN,NaN,Agent Affecting Integumentary System,1,Other therapy
4,166,7-Ethyl-10-Hydroxycamptothecin,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...,C61618,"['SN 38', 'SN-38', '7-Ethyl-10-hydroxy-20(S)-c...",[],"['SN 38', 'SN-38', '7-Ethyl-10-hydroxy-20(S)-c...",3,3,NaN,...,Topoisomerase Inhibitor,Antineoplastic Enzyme Inhibitor,Antineoplastic Protein Inhibitor,Signal Transduction Inhibitor,Antineoplastic Agent,Pharmacologic Substance,"Drug, Food, Chemical or Biomedical Material",Signal Transduction Inhibitor,155,Targeted therapy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
613,27691,WRN Inhibitor HRO761,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...,C210827,"['HRO761', 'HRO 761', 'HRO-761', 'WRN Helicase...",[],"['HRO761', 'HRO 761', 'HRO-761', 'WRN Helicase...",5,5,An orally bioavailable selective and allosteri...,...,Pharmacologic Substance,"Drug, Food, Chemical or Biomedical Material",NaN,NaN,NaN,NaN,NaN,Enzyme Inhibitor,64,Targeted therapy
614,27236,WRN Inhibitor RO7589831,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...,C204483,"['RG6457', 'VVD214', 'RG 6457', 'RG-6457', 'VV...",[],"['RG6457', 'VVD214', 'RG 6457', 'RG-6457', 'VV...",14,14,An orally bioavailable and small molecule inhi...,...,Pharmacologic Substance,"Drug, Food, Chemical or Biomedical Material",NaN,NaN,NaN,NaN,NaN,Enzyme Inhibitor,64,Targeted therapy
618,20008,Ziftomenib,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...,C164227,"['KO539', 'KO 539', 'KO-539', 'Komzifti', 'Men...",[],"['KO539', 'KO 539', 'KO-539', 'Komzifti', 'Men...",8,8,An orally bioavailable inhibitor of the menin-...,...,Antineoplastic Agent,Pharmacologic Substance,"Drug, Food, Chemical or Biomedical Material",NaN,NaN,NaN,NaN,Signal Transduction Inhibitor,171,Targeted therapy
620,577,Zoledronic Acid,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...,C1699,"['Zometa', 'Aclasta', 'Reclast', 'ZOL 446', 'C...",[],"['Zometa', 'Aclasta', 'Reclast', 'ZOL 446', 'C...",9,9,A drug used to treat patients with hypercalcem...,...,Agent Affecting Musculoskeletal System,Pharmacologic Substance,"Drug, Food, Chemical or Biomedical Material",NaN,NaN,NaN,NaN,Agent Affecting Musculoskeletal System,1,Other therapy


## Supplementary figure 1

In [6]:
old_clean_df_step4 = pd.read_csv('../../file_merging/Previous/clean_df_step4.csv')
new_clean_df_step4 = pd.read_csv('../../results_openalex/02_cleaning/clean_df_step4.csv')

In [11]:
clean_df_step4 = pd.concat([old_clean_df_step4, new_clean_df_step4], axis=0, ignore_index=True)
clean_df_step4 = clean_df_step4.drop_duplicates()
clean_df_step4.to_csv('../../file_merging/Updated/clean_df_step4.csv', index=False)
clean_df_step4

,PaperId,PaperTitle,Citations,CoFoS,Authors,Abstract,Language,PubYear,PubDate
0,4405900941,Oncological outcomes following extreme oncopla...,1978649917|2067446979|2118003745|2586578186|26...,71924100|2775897962|530470458|95190672|2781467...,"Megan Chua Wern Ee, A C F Hui, Wade M. Chew, E...",Locally advanced breast cancer (LABC) accounts...,en,2024,2024-12-31
1,4405941037,Regarding: Alpha1 antitrypsin deficiency assoc...,2915717782|3130356516|4248532682|4403027918,71924100|2780535462|72563966|2908647359|126322...,"Malin Fromme, Katharina Remih, Carolin V. Schn...","Dear Editor, We would like to congratulate Kor...",en,2024,2024-12-31
2,4405952152,Incidence and risk factors of immune checkpoin...,2095826876|2107150019|2109653150|2152897456|26...,71924100|61511704|126322002|121608353|27758625...,"Tae Kyun Kim, Hyun Seok Lee, Eun Soo Kim",Immune checkpoint inhibitors (ICIs) are effect...,en,2024,2024-12-31
3,4405971509,Pedicle ossification following mandibular reco...,107451240|1981629435|1989257149|2021927778|205...,71924100|2780037786|199343813|2910884260|27788...,"Jae-Hee Ko, Min Gyeong Kim, Sung Min Kim, Ui H...",Pedicle ossification is a rare but significant...,en,2024,2024-12-31
4,4406120665,Artificial Intelligence for Autonomous Robotic...,1795020324|1997866278|2258079073|2301358467|23...,103203806|199033989|126894567|71924100|4100814...,"Dae Young Lee, Hee Jo Yang",Artificial intelligence (AI) has emerged as a ...,en,2024,2024-12-31
...,...,...,...,...,...,...,...,...,...
2362490,7125285525,Intraoral Lipoma of the Mental Region,2942649268|2778916583|4289551905|2019266211|43...,2776730994|71924100|2777910003|2781197403|1057...,"Shehab Ahmed Hamad, Ahmed Alraad, Osamah Ahmed",Lipomas are the most prevalent mesenchymal tum...,en,2025,2025-01-01
2362491,7125353987,Chemico-physical and biological properties of ...,NaN,185592680|542903549|202751555|136238340|277615...,"Giorgia Miolo, Elisabetta De Diana, Marina Cop...",INTRODUCTION Monoclonal antibodies (mAbs) have...,en,2025,2025-01-01
2362492,7125357929,Neutrophil-derived chitinase 3 -like 1 drives ...,NaN,86803240|8891405|2780380082|136449434|50294259...,"Robbe Salembier, Caro De Haes, Bo Wylin, Krist...",Chitinase-like proteins (CLPs) are key immunos...,en,2025,2025-01-01
2362493,7125358019,Orphine usage in the management of pain by doc...,NaN,2777389121|71924100|2991742891|2983098980|5123...,Nzali Arnold Nzale,Introduction: The World Health Assembly reiter...,en,2025,2025-01-01


## Supplementary figure 3

In [2]:
import pandas as pd

old_stud_cat = pd.read_csv('../../file_merging/Previous/final_gc_classificaton_output_199726.csv')
new_stud_cat = pd.read_csv('../../results_openalex/04_03_classifier/final_gc_classificaton_output_14498.csv')

In [ ]:
stud_cat = pd.concat([old_stud_cat, new_stud_cat], axis=0, ignore_index=True)
stud_cat = stud_cat.drop_duplicates()
stud_cat.to_csv(f'../../file_merging/Updated/final_gc_classificaton_output_{len(stud_cat)}.csv', index=False)
stud_cat

,PaperId,PaperTitle,Abstract,PubYear,Study_design,Study Type,Probability
0,4405900528,Tissue Prior to the Initial Hematoxylin-Eosin ...,"Small biopsies are used for histologic, immuno...",2024.0,In vitro study,NaN,NaN
1,4405900803,Detection of urine circulating tumor DNA using...,Aim: This study aims to evaluate the feasibili...,2024.0,Clinical study,NaN,NaN
2,4405901040,Evaluation of a Novel PLGA-HA-Based Drug Deliv...,Colorectal carcinoma (CRC) is a very important...,2024.0,In vitro study,NaN,NaN
3,4405919948,Gasdermin D regulates the activation of EGFR i...,Gasdermin D (GSDMD) is a key effector molecule...,2024.0,In vitro study,NaN,NaN
4,4405924568,Silencing of Epidermal Growth Factor-like Doma...,Ovarian cancer (OC) is the second most common ...,2024.0,In vitro study,NaN,NaN
...,...,...,...,...,...,...,...
214219,7116128796,Expression of endothelin-1 in unaffected colon...,Introduction: Endothelin-1 is a vasoconstricto...,NaN,Clinical study,-,NaN
214220,7117235300,Time-dependent exposure to venetoclax induces ...,Aim: Neuroblastoma is among the most widely di...,NaN,In vitro study,-,NaN
214221,7118161155,USP3 Promotes Glioma Progression by Stabilizin...,Glioma is an aggressive primary brain tumor wi...,NaN,In vitro study,-,NaN
214222,7120596868,Translational study of IL-33 expression and it...,Triple-negative breast cancer (TNBC) is an agg...,NaN,Clinical study,-,NaN


## Supplementary figure 4

In [1]:
import pandas as pd

old_treatment_mapping_df = pd.read_csv(f'../../file_merging/Previous/treatment_mapping_with_matches.csv')
new_treatment_mapping_df = pd.read_csv(f'../../results_openalex/04_categorization/treatment_mapping_with_matches.csv')
print(f'Previous shape: {old_treatment_mapping_df.shape}')
print(f'New shape: {new_treatment_mapping_df.shape}')

/var/folders/57/yk3w62md44z9m6ndy61mkkz80000gn/T/ipykernel_5983/2778675634.py:3: DtypeWarning: Columns (0: Citations) have mixed types. Specify dtype option on import or set low_memory=False.
  old_treatment_mapping_df = pd.read_csv(f'../../file_merging/Previous/treatment_mapping_with_matches.csv')


Previous shape: (308748, 738)
New shape: (27069, 1344)


/var/folders/57/yk3w62md44z9m6ndy61mkkz80000gn/T/ipykernel_5983/2778675634.py:4: DtypeWarning: Columns (0: CoFoS) have mixed types. Specify dtype option on import or set low_memory=False.
  new_treatment_mapping_df = pd.read_csv(f'../../results_openalex/04_categorization/treatment_mapping_with_matches.csv')


In [2]:
import warnings
warnings.filterwarnings("ignore")

keys = ['PaperId', 'PaperTitle', 'Citations', 'CoFoS', 'Authors', 'Abstract', 'Language', 'PubYear', 'PubDate', 'BioBERT']

merged = old_treatment_mapping_df.merge(
    new_treatment_mapping_df,
    on=keys,
    how='outer',
    suffixes=('_old', '_new')
)

treatment_mapping_df = merged[keys].copy()

# 👉 colonne NON chiave nei dataframe originali
old_cols = set(old_treatment_mapping_df.columns) - set(keys)
new_cols = set(new_treatment_mapping_df.columns) - set(keys)

all_cols = old_cols.union(new_cols)

for col in all_cols:
    old_col = f'{col}_old'
    new_col = f'{col}_new'

    if old_col in merged.columns and new_col in merged.columns:
        # presenti in entrambi → OR logico
        treatment_mapping_df[col] = (
            merged[old_col].fillna(0).astype(bool) |
            merged[new_col].fillna(0).astype(bool)
        ).astype(int)

    elif old_col in merged.columns:
        treatment_mapping_df[col] = merged[old_col].fillna(0).astype(int)

    elif new_col in merged.columns:
        treatment_mapping_df[col] = merged[new_col].fillna(0).astype(int)

    else:
        treatment_mapping_df[col] = merged[col].fillna(0).astype(int)

treatment_mapping_df.to_csv("../../file_merging/Updated/treatment_mapping_with_matches.csv", index=False)

print(f'Shape after merge and process: {treatment_mapping_df.shape}')
treatment_mapping_df

Shape after merge and process: (335817, 1508)


,PaperId,PaperTitle,Citations,CoFoS,Authors,Abstract,Language,PubYear,PubDate,BioBERT,...,MAP2K4,2657,20800,FGFR4 Inhibitor H3B-6527,2622,39,Revumenib,9,Larotrectinib Regimen,CHMFL-KIT-031
0,205112,targeted therapy for genetic cancer syndromes:...,NaN,2780846426|71924100|2777781617|2779134260|2781...,"Rishi Agarwal, S Liebe, Michelle L. Turski, Sm...","von hippel-lindau disease, cowden syndrome, an...",en,2015,2015-02-01,"PIK3CA, MTOR, AKT1",...,0,0,0,0,0,0,0,0,0,0
1,219914,the relationship between mucin phenotype and c...,NaN,179264091|127716648|204232928|207886595|142724...,"Juan Li, Lihua Wu, Yan Shi, Huamin Li, Meiyue ...",gastric hyperplastic polyps (ghps) are the mos...,en,2014,2014-03-01,TP53,...,0,0,0,0,0,0,0,0,0,0
2,382455,gata3 in the urinary bladder: suppression of n...,1521285578|1587151069|1946088674|1974849032|19...,102037460|61367390|555283112|502942594|2910041...,"Yi Li, Hitoshi Ishiguro, Takashi Kawahara, Yur...",recent evidence suggests the involvement of se...,en,2014,2014-01-01,"PTEN, TP53, FGFR3, AR, MYC",...,0,0,0,0,0,0,0,0,0,0
3,828048,current progress and management in molecular t...,NaN,2780333294|2778695046|71924100|2779761222|2780...,"Ritsuko Okamura, Iwao Sugitani",surgery results in improved outcomes for most ...,en,2014,2014-02-01,EGFR,...,0,0,0,0,0,0,0,0,0,0
4,1058396,experimental study on anti-tumor effect and me...,NaN,2777926168|2776415932|29537977|153911025|55318...,"Xiaoliang Liu, Huanqiu Liu, Ji Li, Le Yang, Xi...",to explore anti-cancer effect and mechanism of...,en,2014,2014-11-01,"EGFR, CDK4, CDK6",...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
335812,7125355214,approches immuno-thrapeutiques et immuno-diagn...,NaN,170493617|159654299|502942594|81885089|1855926...,Amaury Herbet,the endothelin axis plays a key role in many p...,en,2025,2025-12-10,"3, 2, 1",...,0,0,0,0,0,0,0,0,0,0
335813,7125357519,ct-based radiomics for non-invasive prediction...,NaN,2778304055|2778019345|71924100|2778559731|1263...,"Wu M, Du Z, Xiao Y, Wang Y., Yang J, Li Z., Li...","meilong wu,1,* zhiyong du,1,* ying xiao,2 yan ...",en,2026,2026-01-01,67,...,0,0,0,0,0,0,0,0,0,0
335814,7125357929,neutrophil-derived chitinase 3 -like 1 drives ...,NaN,86803240|8891405|2780380082|136449434|50294259...,"Robbe Salembier, Caro De Haes, Bo Wylin, Krist...",chitinase-like proteins (clps) are key immunos...,en,2025,2025-01-01,1,...,0,0,0,0,0,0,0,0,0,0
335815,7125359166,risk factors for ileus after radical gastrecto...,NaN,34626388|2777125728|71924100|2780470880|151956...,"Niu M., Qiao Y, Wang B, Zhang J","min niu, yun qiao, bo wang, jinjie zhang depar...",en,2026,2026-01-01,6,...,0,0,0,0,0,0,0,0,0,0


## Supplementary 5

In [1]:
import pandas as pd

old_top_20_df = pd.read_csv("../../file_merging/Previous/Top_20_Variants.csv")
new_top_20_df = pd.read_csv("../../results_openalex/05_LLM_variant_extraction/Top_20_Variants.csv")

top_20_df = pd.merge(old_top_20_df.drop(columns=['Percentage']),
                     new_top_20_df.drop(columns=['Percentage']),
                     on='Variant', how='outer')
top_20_df['Count'] = top_20_df['Count_x'].fillna(0.0) + top_20_df['Count_y'].fillna(0.0)
top_20_df = top_20_df.sort_values(by='Count', ascending=False).head(20)
top_20_df = top_20_df.drop(columns=['Count_x', 'Count_y'])
top_20_df['Percentage'] = (100 * top_20_df['Count'] / (16089 + 354)).round(2)
top_20_df.to_csv("../../file_merging/Updated/Top_20_Variants.csv", index=False)
top_20_df

,Variant,Count,Percentage
22,v600e_BRAF,3919.0,23.83
7,g12d_KRAS,1553.0,9.44
20,t790m_EGFR,1206.0,7.33
12,l858r_EGFR,1111.0,6.76
6,g12c_KRAS,766.0,4.66
9,g12v_KRAS,611.0,3.72
15,r132h_IDH1,461.0,2.80
11,h1047r_PIK3CA,450.0,2.74
3,e545k_PIK3CA,327.0,1.99
10,g13d_KRAS,279.0,1.70


## Supplementary 6

In [8]:
import pandas as pd

old_df_summary = pd.read_csv("../../file_merging/Previous/oncomine_gene_summary_stats_forfigure.csv")
new_df_summary = pd.read_csv("../../results_openalex/05_LLM_variant_extraction/oncomine_gene_summary_stats_forfigure.csv")

# Manually adjust wrong counters
new_df_summary.loc[new_df_summary.index[-1], 'Count'] = 755.0
new_df_summary.loc[new_df_summary.index[-2], 'Count'] = new_df_summary.loc[new_df_summary.index[-1], 'Count'] - new_df_summary.loc[new_df_summary.index[-3], 'Count']

df_summary = pd.merge(old_df_summary.drop(columns=['Percentage']),
                      new_df_summary.drop(columns=['Percentage']),
                      on='Category')
# I manually verified that none of the genes not extracted in the old analysis were extracted in the new, this means
# the next extracted oncomine genes are a subset of the old ones, so we keep the old statistics
df_summary.loc[df_summary.index[:3], 'Count'] = df_summary.loc[df_summary.index[:3], 'Count_x']

# For the other statistics we sum
df_summary.loc[df_summary.index[3:], 'Count'] = df_summary.loc[df_summary.index[3:], 'Count_x'] + df_summary.loc[df_summary.index[3:], 'Count_y']

df_summary = df_summary.drop(columns=['Count_x', 'Count_y'])

df_summary['Percentage'] = 0.0
df_summary.loc[df_summary.index[:3], 'Percentage'] = (df_summary.loc[df_summary.index[:3], 'Count'] / df_summary.iloc[0]['Count'] * 100).round(2)
df_summary.loc[df_summary.index[3:], 'Percentage'] = (df_summary.loc[df_summary.index[3:], 'Count'] / df_summary.iloc[-1]['Count'] * 100).round(2)

df_summary.to_csv("../../file_merging/Updated/oncomine_gene_summary_stats_forfigure.csv")
df_summary

,Category,Count,Percentage
0,Oncomine genes total,161.0,100.00
1,Oncomine genes not extracted,12.0,7.45
2,Oncomine genes extracted,149.0,92.55
3,Oncomine gene mentions,30546.0,84.89
4,Other gene mentions,5439.0,15.11
5,Total gene mentions,35985.0,100.00


## Supplementary 7

In [1]:
import pandas as pd

old_dataset_lengths_df = pd.read_csv("../../file_merging/Previous/conditional_filterting_of_final_datasets_for_coassciaitions_analysis.csv")
new_dataset_lengths_df = pd.read_csv("../../results_openalex/variantscape/conditional_filterting_of_final_datasets_for_coassciaitions_analysis.csv")

dataset_lengths_df = pd.merge(old_dataset_lengths_df.drop(columns=['Percentage']),
                              new_dataset_lengths_df.drop(columns=['Percentage']),
                                                          on='Category')
dataset_lengths_df['Count'] = dataset_lengths_df['Count_x'] + dataset_lengths_df['Count_y']
dataset_lengths_df = dataset_lengths_df.drop(columns=['Count_x', 'Count_y'])
dataset_lengths_df['Percentage'] = dataset_lengths_df['Count'] / dataset_lengths_df.iloc[0]['Count'] * 100
dataset_lengths_df.to_csv("../../file_merging/Updated/conditional_filterting_of_final_datasets_for_coassciaitions_analysis.csv")
dataset_lengths_df

,Category,Count,Percentage
0,Initial Dataset,335817,100.000000
1,Cancer Type Dataset,214226,63.792482
2,Study Design Dataset,214224,63.791887
3,Treatment Dataset,138212,41.156940
4,Variant Dataset,16447,4.897608
5,All conditions TRUE,7423,2.210430


## Supplementary 10 - 11

Solved directly in 07.02 since they depend on dataframe already created there

## Unique variants

In [4]:
import pandas as pd

old_v = pd.read_csv('../../file_merging/Updated/variant_counts_summary.csv')
new_v = pd.read_csv('../../results_openalex/05_LLM_variant_extraction/variant_counts_summary.csv')

len(set(old_v['Unnamed: 0']).union(set(new_v['Unnamed: 0'])))

11108